# Base SmolVLA under three noise seeds on LIBERO-10 task 0: do failures repeat? (Colab, A100)

Runs `hybrid_diffusion/run_base_noise_seeds.sh` with `PARALLEL=1`: the base policy on the same 50 init
states under sampling-noise seeds 1, 2 and 3, as three concurrent processes on one GPU, followed by
`failure_overlap.py`. No head is trained and no file from the local machine is needed. The hypothesis
and decision rule are in `hybrid_diffusion/README.md`.

**Before you start**

Runtime > Change runtime type > **A100 GPU** (High-RAM if offered).

Results and logs go to Google Drive, and the run resumes from them. If the runtime is recycled:

- run not finished: run steps 1 to 5 again and then step 7; finished episodes are not repeated.
- run finished: run steps 1 to 3 and then step 9.

Keep every evaluation of this comparison on Colab. Do not pool these results with episodes run locally.

## 1. Settings

In [ ]:
import json, os, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "main"
REPO = "/content/efficient_vla"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"

DRIVE_DIR = "/content/drive/MyDrive/efficient_vla_runs"
OUT_ROOT = f"{DRIVE_DIR}/task0_base_noise_seeds"   # resumable state lives here
TASK_ID = "0"
SEEDS = "1 2 3"

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"


def sh(cmd, check=True, env=None):
    # Capture and print: Colab does not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")

## 2. GPU and Drive

In [ ]:
sh("nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv")
print("CPU cores:", os.cpu_count())

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_ROOT, exist_ok=True)
print("Output folder:", OUT_ROOT)

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

A separate virtual environment with the exact package versions of the local `lerobot_env`, so the
Colab kernel's own packages are left alone. Takes several minutes (PyTorch and CUDA libraries).

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some Colab images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")
sh(f"uv pip install -q --python {PY} --no-deps -r {REPO}/hybrid_diffusion/requirements-colab.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")   # the run scripts put $HOME/LIBERO on PYTHONPATH
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Run settings and sanity checks

In [ ]:
RUN_ENV = dict(
    os.environ,
    PYTHON_BIN=PY,
    OUT_ROOT=OUT_ROOT,
    TASK_ID=TASK_ID,
    SEEDS=SEEDS,
    MUJOCO_GL="egl",
    PYOPENGL_PLATFORM="egl",
    MPLBACKEND="Agg",          # the kernel's inline backend does not exist in the venv
    WANDB_MODE="offline",      # no W&B login needed
    OMP_NUM_THREADS="2",       # the eval processes share the CPU cores
)

sh(f"{PY} -c \"import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))\"", env=RUN_ENV)
sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('mujoco', mujoco.__version__, '| EGL context ok')\"", env=RUN_ENV)
sh(f"cd {REPO} && {PY} -m pytest tests/ -q", env=RUN_ENV)

## 6. Timing check (3 episodes)

Runs the first three episodes of seed 1. They count towards the final result, so nothing is wasted.
The time includes the one-off model download and a model load.

Check two things before launching the full run:

- The run finishes and prints three outcomes. Failures are expected on this task: the local base policy
  solved 16 of 50 init states, so three failures out of three are not a sign of a broken environment.
  A failed episode runs the full 520 steps.
- Minutes per episode. Total wall-clock is roughly 150 episodes x that figure / 3 processes, if the
  per-episode time holds with three running together.

In [ ]:
t0 = time.time()
sh(f"bash {REPO}/hybrid_diffusion/run_task3_residual.sh", env=dict(RUN_ENV, SEED="1", STAGES="eval_base", EPISODES="3"))
minutes = (time.time() - t0) / 60
eps = json.load(open(f"{OUT_ROOT}/eval_base_s1/eval_results.json"))["episodes"]
for e in eps[:3]:
    print(f"init state {e['seed']}: success={e['success']} steps={e['steps']}")
print(f"{minutes:.1f} min for 3 episodes ({minutes / 3:.1f} min per episode, including model load)")
print("If this cell was re-run, the episodes were already done and the timing is meaningless.")

## 7. Launch the full run

Starts in the background and returns immediately. Running it again while a run is active does nothing.
After a disconnect or a new runtime, run this cell again to resume.

In [ ]:
running = subprocess.run(["pgrep", "-f", "run_base_noise_seeds"], capture_output=True, text=True).stdout.split()
if running:
    print("A run is already active (pids:", " ".join(running) + "). Not starting another.")
else:
    log = f"{OUT_ROOT}/base_noise_seeds.runner.log"
    subprocess.Popen(
        f"nohup bash {REPO}/hybrid_diffusion/run_base_noise_seeds.sh >> '{log}' 2>&1 &",
        shell=True, executable="/bin/bash", env=dict(RUN_ENV, PARALLEL="1"), start_new_session=True,
    )
    print("Started. Log:", log)

## 8. Progress (re-run any time)

Needs only steps 1 and 2. The GPU stays at 0 MiB for the first few minutes after a launch while the
processes load the model.

In [ ]:
for seed in SEEDS.split():
    p = Path(OUT_ROOT, f"eval_base_s{seed}", "eval_results.json")
    eps = json.load(open(p))["episodes"] if p.exists() else []
    print(f"seed {seed}: {len(eps):2d}/50 episodes, {sum(e['success'] for e in eps)} successes")

running = subprocess.run(["pgrep", "-f", "run_base_noise_seeds"], capture_output=True, text=True).stdout.split()
print("run active:", bool(running))
sh("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader", check=False)
sh(f"tail -n 8 '{OUT_ROOT}/base_noise_seeds.runner.log'", check=False)

## 9. Results

The run writes `failure_overlap_s1-2-3.json` when all three evaluations are complete. This cell
recomputes it from the per-episode records and fails if any evaluation is short of 50 episodes.

The analysis uses only the standard library, so it runs on Colab's own Python. On a new runtime run
steps 1 to 3 and then this cell; the environment of step 4 is not needed.

In [ ]:
sh(f"python3 -S {REPO}/hybrid_diffusion/failure_overlap.py --out_root '{OUT_ROOT}' --seeds {SEEDS} --episodes 50")
sh(f"cat '{OUT_ROOT}/provenance_base_noise_seeds.txt'", check=False)

## 10. Download the results for the repo

Zips the evaluation folders, the summary, the provenance file and the logs (well under 1 MB). Unpack it
into `week6/task0_base_noise_seeds_colab/` in the repo, not into `hybrid_diffusion/outputs/`.

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /content/task0_base_noise_seeds_colab.zip "
   "eval_base_s* failure_overlap_*.json provenance_base_noise_seeds.txt *.log")
sh("du -h /content/task0_base_noise_seeds_colab.zip")
from google.colab import files
files.download("/content/task0_base_noise_seeds_colab.zip")